# PHASE 4 — Crossed random effects

## The model in the manuscript omits the image effect

Section 5.3 specified $d_{if} = \mu + a_f + e_{if}$. There is no image term, yet the same 900
images appear under every corruption family --- which is precisely why an image-clustered
bootstrap is needed. The correct specification is **crossed**: $d_{if} = \mu + a_f + b_i + e_{if}$,
under which

$$\frac{\mathrm{SE}_{\rm family}}{\mathrm{SE}_{\rm image}}
= \sqrt{\frac{m\,\sigma_a^2 + \sigma_e^2}{F\,\sigma_b^2 + \sigma_e^2}}$$

and the original Equation (2) is that expression with $\sigma_b^2$ set to zero.

## Why it matters, and why it is not fatal

Simulated at $m=900$, $F=6$, $\sigma_a^2 = 0$: the ratio is 1.19 at $\sigma_b^2=0$, but 0.98, 0.62
and 0.47 at $\sigma_b^2 = 0.1$, $0.5$ and $1.0$. **With an image effect present and no family
effect, the family SE falls *below* the image SE.** The identity "the two coincide exactly" is
false, and the manuscript has been restated.

The direction is favourable: an image effect can only push the ratio down, so an observed ratio
above one remains conservative evidence of family heterogeneity. What is *not* established is the
magnitude. Because `var_e` was estimated as $\mathrm{SE}_{\rm image}^2 m F$, it actually captures
$F\sigma_b^2 + \sigma_e^2$; the $\sigma_e$ column of Table 5 is not the residual SD, and $\rho$
carries the image variance in its denominator. In simulation the bias flips sign with
$\sigma_b^2/\sigma_e^2$, so its direction on real data must be measured, not assumed.

## What this notebook does

Fits the crossed model by balanced-ANOVA moment matching on the complete image $\times$ family
difference matrix --- exact for this design, no solver, no extra dependency. It reports the
two-component $\rho$ the manuscript quotes beside the crossed
$\rho_f = \sigma_a^2/(\sigma_a^2+\sigma_b^2+\sigma_e^2)$, and checks the exact ratio formula
against the observed bootstrap. The estimator is validated on synthetic data with known components
before being applied to anything real.

A few minutes; extraction is reused from cache or rebuilt if absent.

## 0. Core module

In [ ]:
CORE_V2 = r"""
import math, hashlib
import numpy as np, cv2

def _clip(x): return np.clip(x, 0, 1)
def _mask3(mask, x): return mask[..., None] if x.ndim == 3 else mask

# ------------------------------------------------------------------ optics --
def defocus_blur(x, s):
    r = [1, 2, 3, 5, 7][s-1]
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2*r+1, 2*r+1)).astype(np.float32)
    return _clip(cv2.filter2D(x, -1, k/k.sum()))

def motion_blur(x, s):
    ksz = [5, 9, 13, 19, 25][s-1]
    ang = np.random.uniform(0, 180)
    k = np.zeros((ksz, ksz), np.float32); k[ksz//2, :] = 1.0
    M = cv2.getRotationMatrix2D((ksz/2-.5, ksz/2-.5), ang, 1.0)
    k = cv2.warpAffine(k, M, (ksz, ksz))
    return _clip(cv2.filter2D(x, -1, k/k.sum()))

def vignetting(x, s):
    st = [0.15, 0.30, 0.45, 0.62, 0.80][s-1]
    h, w = x.shape[:2]; yy, xx = np.mgrid[0:h, 0:w]
    r = np.sqrt(((xx-w/2)/(w/2))**2 + ((yy-h/2)/(h/2))**2)
    m = np.clip(1 - st*np.clip(r-0.4, 0, None)/0.6, 0, 1).astype(np.float32)
    return _clip(x * _mask3(m, x))

def lens_contamination(x, s):
    n = [3, 7, 13, 22, 34][s-1]; h, w = x.shape[:2]
    blurred = cv2.GaussianBlur(x, (0, 0), sigmaX=max(1.0, min(h, w)/40))
    mask = np.zeros((h, w), np.float32)
    for _ in range(n):
        c = (np.random.randint(0, w), np.random.randint(0, h))
        rad = np.random.randint(max(2, int(0.015*w)), max(4, int(0.07*w)))
        cv2.circle(mask, c, rad, 1.0, -1, lineType=cv2.LINE_AA)
    mask = cv2.GaussianBlur(mask, (0, 0), sigmaX=max(1.0, min(h, w)/60))
    m = _mask3(mask, x)                       # guard: 2-D input used to broadcast to (H,W,W)
    return _clip((x*(1-m) + blurred*m) * (1 - 0.25*m))

def vibration_jitter(x, s):
    amp = [0.6, 1.4, 2.6, 4.2, 6.5][s-1]; h, w = x.shape[:2]
    dx, dy = np.random.uniform(-amp, amp, 2)
    out = cv2.warpAffine(x, np.float32([[1, 0, dx], [0, 1, dy]]), (w, h),
                         flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT_101)
    ksz = int(max(3, 2*round(amp)+1))
    k = np.zeros((ksz, ksz), np.float32); k[ksz//2, :] = 1.0
    M2 = cv2.getRotationMatrix2D((ksz/2-.5, ksz/2-.5), math.degrees(math.atan2(dy, dx)), 1.0)
    k = cv2.warpAffine(k, M2, (ksz, ksz))
    return _clip(cv2.filter2D(out, -1, k/max(k.sum(), 1e-8)))

# ------------------------------------------------------------------ sensor --
def gaussian_noise(x, s):
    return _clip(x + np.random.normal(0, [0.03, 0.06, 0.10, 0.16, 0.24][s-1], x.shape))

def shot_noise(x, s):
    lam = [80, 35, 15, 7, 3][s-1]
    return _clip(np.random.poisson(x*lam)/float(lam))

def scanline_banding(x, s):
    amp = [0.03, 0.06, 0.11, 0.17, 0.25][s-1]; h = x.shape[0]
    period = np.random.uniform(3, 22); phase = np.random.uniform(0, 2*np.pi)
    b = (1 + amp*np.sin(2*np.pi*np.arange(h)/period + phase)).astype(np.float32)
    return _clip(x * (b[:, None, None] if x.ndim == 3 else b[:, None]))

# ------------------------------------------------------------- photometric --
def illumination_gradient(x, s):
    st = [0.10, 0.20, 0.32, 0.46, 0.62][s-1]; h, w = x.shape[:2]
    ang = np.random.uniform(0, 2*np.pi); yy, xx = np.mgrid[0:h, 0:w]
    u = (xx/w-.5)*np.cos(ang) + (yy/h-.5)*np.sin(ang)
    g = (1 + st*u/(np.abs(u).max()+1e-8)).astype(np.float32)
    return _clip(x * _mask3(g, x))

def brightness_drift(x, s):
    # v2: gamma instead of an additive offset. Gamma maps [0,1] -> [0,1] and CANNOT clip.
    # v1 saturated 36.8% of pixels at severity 5 on Magnetic Tile, which destroyed dynamic
    # range rather than shifting brightness and made the top of the ladder meaningless.
    g = [1.12, 1.26, 1.45, 1.72, 2.05][s-1]
    if np.random.rand() < 0.5: g = 1.0/g
    return _clip(np.power(_clip(x), g))

def contrast_loss(x, s):
    g = [0.80, 0.65, 0.50, 0.36, 0.24][s-1]
    m = x.mean(axis=(0, 1), keepdims=True)
    return _clip((x-m)*g + m)

# ---------------------------------------------------------------- pipeline --
def jpeg_compression(x, s):
    q = [70, 50, 32, 18, 9][s-1]
    src = (x[..., ::-1]*255).astype(np.uint8) if x.ndim == 3 else (x*255).astype(np.uint8)
    _, enc = cv2.imencode(".jpg", src, [int(cv2.IMWRITE_JPEG_QUALITY), q])
    dec = cv2.imdecode(enc, cv2.IMREAD_COLOR if x.ndim == 3 else cv2.IMREAD_GRAYSCALE)
    return (dec[..., ::-1] if x.ndim == 3 else dec).astype(np.float32)/255.

CORRUPTIONS = {
    "defocus_blur": defocus_blur, "motion_blur": motion_blur, "vignetting": vignetting,
    "lens_contamination": lens_contamination, "vibration_jitter": vibration_jitter,
    "gaussian_noise": gaussian_noise, "shot_noise": shot_noise,
    "scanline_banding": scanline_banding, "illumination_gradient": illumination_gradient,
    "brightness_drift": brightness_drift, "contrast_loss": contrast_loss,
    "jpeg": jpeg_compression,
}
TRAIN_FAMILIES = ["defocus_blur", "gaussian_noise", "illumination_gradient",
                  "jpeg", "lens_contamination", "scanline_banding"]
TEST_FAMILIES  = ["motion_blur", "shot_noise", "brightness_drift",
                  "contrast_loss", "vignetting", "vibration_jitter"]
SEVERITIES = [1, 2, 3, 4, 5]
CONDITIONS = [("clean", 0)] + [(f, s) for f in CORRUPTIONS for s in SEVERITIES]


def corruption_seed(image_id, family, severity=None):
    # Depends on (image_id, family) only, NOT severity: nuisance parameters (blur angle,
    # banding period, blob positions, gradient orientation) stay fixed so that severity is
    # the sole varying factor. Seeding on severity too made scanline_banding score 0.20.
    h = hashlib.sha256(f"{image_id}|{family}".encode()).digest()
    return int.from_bytes(h[:4], "little")


def apply_corruption(img_u8, family, severity, image_id=None, seed=None):
    if family == "clean":
        return img_u8
    if seed is None and image_id is not None:
        seed = corruption_seed(image_id, family)
    st = None
    if seed is not None:
        st = np.random.get_state(); np.random.seed(seed % (2**32))
    try:
        out = CORRUPTIONS[family](img_u8.astype(np.float32)/255., severity)
    finally:
        if st is not None: np.random.set_state(st)
    return (np.clip(out, 0, 1)*255).astype(np.uint8)


# ------------------------------------------------------- quality descriptors -
def _gray(im):
    g = cv2.cvtColor(im, cv2.COLOR_RGB2GRAY) if im.ndim == 3 else im
    return g.astype(np.float32)/255.

def _sharp(im): return float(cv2.Laplacian(_gray(im), cv2.CV_32F).var())

def _noise(im):
    g = _gray(im); h, w = g.shape
    M = np.array([[1, -2, 1], [-2, 4, -2], [1, -2, 1]], np.float32)
    return float(np.abs(cv2.filter2D(g, -1, M)).sum()*math.sqrt(math.pi/2) /
                 (6*max(w-2, 1)*max(h-2, 1)))

def _block(im):
    dh = np.abs(np.diff(_gray(im), axis=1))
    on = dh[:, 7::8].mean() if dh.shape[1] > 8 else 0.0
    return float(on/(dh.mean()+1e-8) - 1.0)

def _hf(im):
    g = _gray(im); f = np.abs(np.fft.fftshift(np.fft.fft2(g))); h, w = g.shape
    cy, cx = h//2, w//2; r = max(4, min(h, w)//8)
    return float(1.0 - f[cy-r:cy+r, cx-r:cx+r].sum()/(f.sum()+1e-8))


def quality_descriptor(img_u8):
    # 8-d ABSOLUTE no-reference descriptor (unchanged from v1).
    g = _gray(img_u8); h, w = g.shape
    if img_u8.ndim == 3:
        rg = img_u8[..., 0].astype(np.float32) - img_u8[..., 1]
        yb = .5*(img_u8[..., 0].astype(np.float32) + img_u8[..., 1]) - img_u8[..., 2]
        colour = float((np.sqrt(rg.std()**2 + yb.std()**2)
                        + .3*np.sqrt(rg.mean()**2 + yb.mean()**2))/255.)
    else:
        colour = 0.0
    cen = g[h//4:3*h//4, w//4:3*w//4]
    per = (g.sum()-cen.sum())/max(g.size-cen.size, 1)
    v = np.array([math.log1p(max(_sharp(img_u8), 0.)*1e3), _hf(img_u8),
                  math.log1p(max(_noise(img_u8), 0.)*1e3), _block(img_u8),
                  float(g.mean()), float(g.std()), colour,
                  float(cen.mean()/(per+1e-8))], np.float32)
    return np.nan_to_num(v, nan=0., posinf=0., neginf=0.)


def quality_descriptor_relative(img_u8):
    # 8-d PERTURBATION-RESPONSE descriptor: how much does a statistic move when a known
    # perturbation is applied? Higher severity signal than the absolute set (0.324 vs 0.236
    # on the synthetic benchmark) but NO reduction in class leakage on its own -- ratios
    # remove the absolute texture level, not the spectral shape. Use with class-conditional
    # standardisation, never alone.
    g8 = (_gray(img_u8)*255).astype(np.uint8); eps = 1e-8
    b = cv2.GaussianBlur(g8, (0, 0), 1.5)
    dn = cv2.resize(cv2.resize(g8, (max(2, g8.shape[1]//2), max(2, g8.shape[0]//2)),
                               interpolation=cv2.INTER_AREA),
                    (g8.shape[1], g8.shape[0]), interpolation=cv2.INTER_LINEAR)
    rn = np.random.default_rng(12345)
    nz = np.clip(g8.astype(np.float32) + rn.normal(0, 12, g8.shape), 0, 255).astype(np.uint8)
    _, e = cv2.imencode(".jpg", g8, [int(cv2.IMWRITE_JPEG_QUALITY), 40])
    jp = cv2.imdecode(e, cv2.IMREAD_GRAYSCALE)
    kh = np.zeros((9, 9), np.float32); kh[4, :] = 1/9.
    hb = cv2.filter2D(g8.astype(np.float32), -1, kh).astype(np.uint8)
    vb = cv2.filter2D(g8.astype(np.float32), -1, kh.T).astype(np.uint8)
    x = g8.astype(np.float32)/255.
    v = np.array([
        math.log((_sharp(g8)+eps)/(_sharp(b)+eps)),
        math.log((_sharp(g8)+eps)/(_sharp(dn)+eps)),
        math.log((_noise(nz)+eps)/(_noise(g8)+eps)),
        _block(jp) - _block(g8),
        float(((x+0.25) > 1.0).mean() + ((x-0.25) < 0.0).mean()),
        abs(math.log((_sharp(hb)+eps)/(_sharp(vb)+eps))),
        math.log((_hf(g8)+eps)/(_hf(b)+eps)),
        math.log((_gray(g8).std()+eps)/(_gray(b).std()+eps)),
    ], np.float32)
    return np.nan_to_num(v, nan=0., posinf=0., neginf=0.)


QUALITY_NAMES = ["sharpness", "hf_energy", "noise", "blockiness",
                 "luminance_mean", "luminance_std", "colourfulness", "vignette_ratio"]
RELATIVE_NAMES = ["blur_headroom", "resolution_headroom", "noise_headroom",
                  "compression_headroom", "clipping_headroom", "blur_anisotropy",
                  "hf_retention", "contrast_retention"]
QUALITY_DIM = 8


class ClassConditionalStandardiser:
    # z = (q - mu_c) / sigma_c, with mu_c and sigma_c estimated on DEVELOPMENT data only.
    #
    # Rationale: degradation is relative. A blurry-looking crazing image and a sharp-looking
    # patches image can have identical absolute sharpness; what makes one degraded is that it
    # is blurrier than crazing images normally are. Removing the class-conditional mean strips
    # the content component and leaves the deviation-from-typical -- which is the degradation.
    #
    # At test time c is the model's own prediction. There is no feedback loop: a per-image
    # scalar temperature cannot change the argmax, so the prediction is fixed before the
    # calibrator runs.
    def __init__(self, n_classes):
        self.n_classes = n_classes; self.mu = None; self.sd = None

    def fit(self, q, y):
        d = q.shape[1]
        self.mu = np.zeros((self.n_classes, d), np.float32)
        self.sd = np.ones((self.n_classes, d), np.float32)
        gm, gs = q.mean(0), q.std(0) + 1e-6
        for c in range(self.n_classes):
            m = (y == c)
            if m.sum() >= 5:                 # fall back to global stats for tiny classes
                self.mu[c] = q[m].mean(0); self.sd[c] = q[m].std(0) + 1e-6
            else:
                self.mu[c] = gm; self.sd[c] = gs
        return self

    def transform(self, q, y_pred):
        y_pred = np.asarray(y_pred).astype(int)
        return ((q - self.mu[y_pred]) / self.sd[y_pred]).astype(np.float32)
"""
print(f"embedded core module: {len(CORE_V2.splitlines())} lines")

embedded core module: 246 lines


## 1. Setup and cached features

In [ ]:
#@title Dependencies and data
import subprocess, sys
subprocess.run([sys.executable,"-m","pip","install","-q","timm==1.0.11","scikit-learn==1.5.2",
                "opencv-python-headless==4.10.0.84","pandas==2.2.3","kagglehub","tqdm",
                "setuptools"],check=True)
import os, json, math, time, random, hashlib, warnings
from collections import defaultdict, Counter
from pathlib import Path
import numpy as np, pandas as pd, cv2
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import timm
warnings.filterwarnings("ignore")
SEED=20260821
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
ROOT=Path("/content/sdic"); OUT=ROOT/"phase3"; OUT.mkdir(parents=True,exist_ok=True)
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()
sys.path.insert(0,str(ROOT))
_t=ROOT/"sdic_core_v2.py"; _w=hashlib.sha256(CORE_V2.encode()).hexdigest()
if not _t.exists() or hashlib.sha256(_t.read_text().encode()).hexdigest()!=_w: _t.write_text(CORE_V2)
import importlib, sdic_core_v2; importlib.reload(sdic_core_v2)
from sdic_core_v2 import (TRAIN_FAMILIES, TEST_FAMILIES, SEVERITIES, apply_corruption,
                          quality_descriptor)

import kagglehub
NEU_ROOT=Path(kagglehub.dataset_download("kaustubhdikshit/neu-surface-defect-database"))
neu=[p for p in sorted(NEU_ROOT.rglob("*.jpg"))
     if p.parent.name.lower() not in {"train","validation","images","annotations"}]
ny=np.array([p.parent.name for p in neu]); LUT={n:i for i,n in enumerate(sorted(set(ny)))}
NEU_y=np.array([LUT[l] for l in ny]); NC=len(LUT)
def rd(p): return cv2.cvtColor(cv2.imread(str(p)),cv2.COLOR_BGR2RGB)
set_seed()
per=defaultdict(list)
for p,yy in zip(neu,NEU_y): per[yy].append(p)
sel=[]
for c,v in per.items():
    sel += [neu.index(v[t]) for t in
            np.random.default_rng(SEED+c).choice(len(v),min(150,len(v)),replace=False)]
sel=np.array(sorted(sel)); S_paths=[neu[i] for i in sel]; S_y=NEU_y[sel]

FEAT1=ROOT/"feats_phase1"; FEAT2=ROOT/"feats_phase2"; FEAT3=ROOT/"feats_phase3"
FEAT3.mkdir(parents=True,exist_ok=True)
BACK_FULL="resnet50.a1_in1k"; BACK="resnet50"
SDIC_MAIN=[("clean",0)]+[(f,s) for f in TRAIN_FAMILIES for s in (1,3,5)] \
                       +[(f,s) for f in TEST_FAMILIES for s in SEVERITIES]
def _find(name):
    for d in (FEAT1,FEAT2,FEAT3):
        if (d/name).exists(): return d/name
    return None
def load(fam,sev):
    fp=_find(f"sdic__main__{fam}__{sev}.npz")
    assert fp is not None, (f"condition {fam}/{sev} absent after ensure_features(); "
                            f"this should be unreachable -- check FEAT3 is writable")
    z=np.load(fp); return z[f"f_{BACK}"], z["q"]
from timm.data import resolve_model_data_config
_INTERP={"bilinear":cv2.INTER_LINEAR,"bicubic":cv2.INTER_CUBIC,
         "nearest":cv2.INTER_NEAREST,"area":cv2.INTER_AREA}

def ensure_features():
    """Extract whatever is missing. This notebook does not require a previous session:
    depending on one is how PHASE3 and PHASE3_2 both died on their second cell."""
    miss=[c for c in SDIC_MAIN if _find(f"sdic__main__{c[0]}__{c[1]}.npz") is None]
    print(f"{len(S_paths)} images | cached {len(SDIC_MAIN)-len(miss)}/{len(SDIC_MAIN)} conditions")
    if not miss: return
    print(f"extracting {len(miss)} missing conditions with {BACK} (~9 min for a full rebuild)")
    m=timm.create_model(BACK_FULL,pretrained=True,num_classes=0).eval().to(DEVICE)
    c=resolve_model_data_config(m)
    cfg={"mean":np.array(c["mean"],np.float32),"std":np.array(c["std"],np.float32),"size":224,
         "interp":_INTERP.get(c["interpolation"],cv2.INTER_CUBIC),
         "crop_pct":float(c.get("crop_pct") or 1.0)}
    def prep(im):
        sz=cfg["size"]; to=int(round(sz/cfg["crop_pct"])); h,w=im.shape[:2]; s=to/min(h,w)
        r=cv2.resize(im,(max(1,int(round(w*s))),max(1,int(round(h*s)))),interpolation=cfg["interp"])
        hh,ww=r.shape[:2]; t,l=(hh-sz)//2,(ww-sz)//2
        x=(r[t:t+sz,l:l+sz].astype(np.float32)/255.-cfg["mean"])/cfg["std"]
        return torch.from_numpy(x).permute(2,0,1)
    t0=time.time()
    with torch.no_grad():
        for fam,sev in tqdm(miss,desc="extract"):
            Fs,Qs=[],[]
            for i in range(0,len(S_paths),64):
                imgs=[]
                for pth in S_paths[i:i+64]:
                    im=rd(pth)
                    if fam!="clean": im=apply_corruption(im,fam,sev,image_id=pth.stem)
                    Qs.append(quality_descriptor(im)); imgs.append(im)
                with torch.autocast("cuda",enabled=DEVICE=="cuda"):
                    Fs.append(m(torch.stack([prep(im) for im in imgs]).to(DEVICE))
                              .float().cpu().numpy())
            np.savez_compressed(FEAT3/f"sdic__main__{fam}__{sev}.npz",
                                q=np.stack(Qs), **{f"f_{BACK}":np.concatenate(Fs)})
    del m; torch.cuda.empty_cache()
    print(f"extraction finished in {(time.time()-t0)/60:.1f} min")

ensure_features()

100%|██████████| 26.4M/26.4M [00:00<00:00, 106MB/s] 

Extracting files...


900 images | cached 0/49 conditions
extracting 49 missing conditions with resnet50 (~9 min for a full rebuild)


model.safetensors: reconstructing file:   0%|          |  0.00B /  102MB            

model.safetensors: downloading bytes:           |  0.00B            

extract:   0%|          | 0/49 [00:00<?, ?it/s]

extraction finished in 6.1 min


## 2. Crossed variance components

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, train_test_split

EPS=1e-2
class ScalarT(nn.Module):
    def __init__(self,d=None):
        super().__init__(); self.log_t=nn.Parameter(torch.zeros(()))
    def temperature(self,q): return self.log_t.exp().expand(q.shape[0])+EPS
    def forward(self,lg,q): return lg/self.temperature(q).unsqueeze(-1)
class LinearT(nn.Module):
    def __init__(self,d):
        super().__init__()
        self.register_buffer("mu",torch.zeros(d)); self.register_buffer("sd",torch.ones(d))
        self.lin=nn.Linear(d,1); nn.init.zeros_(self.lin.weight)
        nn.init.constant_(self.lin.bias,math.log(math.exp(1.0-EPS)-1.0))
    def fit_norm(self,q): self.mu.copy_(q.mean(0)); self.sd.copy_(q.std(0).clamp_min(1e-6))
    def temperature(self,q): return F.softplus(self.lin((q-self.mu)/self.sd).squeeze(-1))+EPS
    def forward(self,lg,q): return lg/self.temperature(q).unsqueeze(-1)
def fit_cal(cls,L,Q,Y,epochs=400,lr=1e-2):
    set_seed()
    L=torch.as_tensor(L,dtype=torch.float32); Q=torch.as_tensor(Q,dtype=torch.float32)
    Y=torch.as_tensor(Y,dtype=torch.long); m=cls(Q.shape[1])
    if hasattr(m,"fit_norm"): m.fit_norm(Q)
    o=torch.optim.Adam(m.parameters(),lr=lr)
    for _ in range(epochs):
        o.zero_grad(); F.cross_entropy(m(L,Q),Y).backward(); o.step()
    return m.eval()
def softmax(z):
    z=z-z.max(1,keepdims=True); e=np.exp(z); return e/e.sum(1,keepdims=True)
def nll_pi(p,y): return -np.log(np.clip(p[np.arange(len(y)),y],1e-12,None))

def paired_matrix(A="C4",B="C2",n_folds=5):
    """D[image, family] = mean paired NLL difference over that family's severities."""
    Fc,Qc=load(("clean",0)[0],0); n=len(S_y)
    skf=StratifiedKFold(n_folds,shuffle=True,random_state=SEED)
    FOLDS=[te for _,te in skf.split(np.zeros(n),S_y)]
    D=np.full((n,len(TEST_FAMILIES)),np.nan)
    cal=[("clean",0)]+[(f,s) for f in TRAIN_FAMILIES for s in (1,3,5)]
    for k in range(n_folds):
        te=FOLDS[k]; rest=np.concatenate([FOLDS[j] for j in range(n_folds) if j!=k])
        tr,va=train_test_split(rest,test_size=0.25,random_state=SEED+k,stratify=S_y[rest])
        pr=make_pipeline(StandardScaler(),
                         LogisticRegression(max_iter=4000,class_weight="balanced")).fit(Fc[tr],S_y[tr])
        def lg(Fx):
            d=pr.decision_function(Fx); return d if d.ndim>1 else np.stack([-d,d],1)
        def gather(ix,conds):
            L,Q,Y=[],[],[]
            for fam,sev in conds:
                Fx,Qx=load(fam,sev)
                L.append(lg(Fx[ix])); Q.append(Qx[ix]); Y.append(S_y[ix])
            L=np.concatenate(L); Q=np.concatenate(Q); Y=np.concatenate(Y)
            H=np.zeros((len(L),NC),np.float32); H[np.arange(len(L)),L.argmax(1)]=1.
            return L,Q,H,Y
        Lf,Qf,Hf,Yf=gather(va,cal)
        arms={"C2":(fit_cal(ScalarT,Lf,Qf,Yf),"q"),"C4":(fit_cal(LinearT,Lf,Qf,Yf),"q"),
              "C5":(fit_cal(LinearT,Lf,Hf,Yf),"h")}
        for fi,fam in enumerate(TEST_FAMILIES):
            Lt,Qt,Ht,Yt=gather(te,[(fam,s) for s in SEVERITIES]); ft={"q":Qt,"h":Ht}
            out={}
            for nm in (A,B):
                mdl,f_=arms[nm]
                with torch.no_grad():
                    p=softmax(mdl(torch.as_tensor(Lt,dtype=torch.float32),
                                  torch.as_tensor(ft[f_],dtype=torch.float32)).numpy())
                out[nm]=nll_pi(p,Yt).reshape(len(SEVERITIES),len(te)).mean(0)
            D[te,fi]=out[A]-out[B]
    assert not np.isnan(D).any(), "every image must land in exactly one test fold"
    return D

In [ ]:
def crossed_components(D):
    """Moment estimator for d_if = mu + a_f + b_i + e_if on a complete m x F matrix.

    The design is balanced and complete, so the two-way ANOVA identities are exact:
      E[MS_A] = sigma_e^2 + m*sigma_a^2      (family direction)
      E[MS_B] = sigma_e^2 + F*sigma_b^2      (image direction)
      E[MS_E] = sigma_e^2                    (residual)
    No iterative solver, no extra dependency.
    """
    m,Fn=D.shape
    gm=D.mean(); rm=D.mean(1); cm=D.mean(0)
    MS_A=m*((cm-gm)**2).sum()/(Fn-1)
    MS_B=Fn*((rm-gm)**2).sum()/(m-1)
    resid=D-rm[:,None]-cm[None,:]+gm
    MS_E=(resid**2).sum()/((m-1)*(Fn-1))
    s2_e=MS_E; s2_a=max((MS_A-MS_E)/m,0.0); s2_b=max((MS_B-MS_E)/Fn,0.0)
    tot=s2_a+s2_b+s2_e
    return dict(sigma_a=math.sqrt(s2_a), sigma_b=math.sqrt(s2_b), sigma_e=math.sqrt(s2_e),
                rho_family=s2_a/tot if tot>0 else 0.0,
                rho_image=s2_b/tot if tot>0 else 0.0,
                ratio_exact=math.sqrt((m*s2_a+s2_e)/(Fn*s2_b+s2_e)),
                ratio_2comp=math.sqrt(1+m*s2_a/s2_e) if s2_e>0 else float("inf"))

print("estimator validation on synthetic data (true -> recovered variance)")
_r=np.random.default_rng(0)
for s2a,s2b,s2e in [(0.00,0.00,1.00),(0.00,0.50,1.00),(0.08,0.00,1.00),
                    (0.08,0.50,1.00),(0.30,0.20,0.50)]:
    a=_r.normal(0,math.sqrt(s2a),6); b=_r.normal(0,math.sqrt(s2b),900)
    Ds=a[None,:]+b[:,None]+_r.normal(0,math.sqrt(s2e),(900,6))
    c=crossed_components(Ds)
    print(f"  a {s2a:.2f}->{c['sigma_a']**2:6.3f} | b {s2b:.2f}->{c['sigma_b']**2:6.3f} "
          f"| e {s2e:.2f}->{c['sigma_e']**2:6.3f} | ratio {c['ratio_exact']:6.2f}")
    assert abs(c["sigma_e"]**2-s2e)<0.08 and abs(c["sigma_b"]**2-s2b)<0.08, "component not recovered"
print("  all three components recovered\n")

estimator validation on synthetic data (true -> recovered variance)
  a 0.00-> 0.000 | b 0.00-> 0.000 | e 1.00-> 1.000 | ratio   1.00
  a 0.00-> 0.000 | b 0.50-> 0.508 | e 1.00-> 0.985 | ratio   0.55
  a 0.08-> 0.111 | b 0.00-> 0.000 | e 1.00-> 0.990 | ratio  10.08
  a 0.08-> 0.031 | b 0.50-> 0.459 | e 1.00-> 1.018 | ratio   2.76
  a 0.30-> 0.266 | b 0.20-> 0.183 | e 0.50-> 0.500 | ratio  12.27
  all three components recovered



In [ ]:
#@title Fit the crossed model on the real comparisons
rows=[]
for A,B in [("C4","C2"),("C4","C5")]:
    D=paired_matrix(A,B); c=crossed_components(D)
    rng=np.random.default_rng(SEED)
    flat=D.ravel(); ims=np.repeat(np.arange(D.shape[0]),D.shape[1])
    uq=np.unique(ims); by={u:np.where(ims==u)[0] for u in uq}
    bs=np.array([flat[np.concatenate([by[u] for u in rng.choice(uq,len(uq),True)])].mean()
                 for _ in range(4000)])
    se_img=(np.quantile(bs,.975)-np.quantile(bs,.025))/(2*1.96)
    se_fam=math.sqrt(D.mean(0).var(ddof=1)/D.shape[1])
    rows.append(dict(comparison=f"{A} vs {B}",**{k:round(v,4) for k,v in c.items()},
                     ratio_observed=round(se_fam/se_img,3)))
R=pd.DataFrame(rows); display(R); R.to_csv(OUT/"phase4_crossed.csv",index=False)

print("\nCROSSED DECOMPOSITION")
for _,r in R.iterrows():
    print(f"\n  {r['comparison']}")
    print(f"    sigma_a {r['sigma_a']:.4f}   sigma_b {r['sigma_b']:.4f}   sigma_e {r['sigma_e']:.4f}")
    print(f"    rho_family (crossed) {r['rho_family']:.4f}   rho_image {r['rho_image']:.4f}")
    print(f"    ratio -- exact {r['ratio_exact']:.2f} | two-component {r['ratio_2comp']:.2f} "
          f"| observed {r['ratio_observed']:.2f}")
    gap=abs(r['ratio_exact']-r['ratio_observed'])
    if gap>0.15*max(r['ratio_observed'],1):
        print(f"    WARNING: exact formula and observed bootstrap differ by {gap:.2f};")
        print( "    the balanced-ANOVA assumptions may not hold -- use a proper mixed-model fit")

mi=R.rho_image.max()
print(f"\nrho_image ranges to {mi:.4f}")
if mi<0.02:
    print("  -> negligible. The two-component reduction was harmless; Tables 5-7 stand.")
else:
    print("  -> material. The reported rho is a LOWER BOUND on the family ICC, because the")
    print("     image variance sat in its denominator. Report rho_family as the headline and")
    print("     keep the two-component value for comparability, saying which is which.")
    print("     This makes the effect LARGER than the manuscript currently claims.")
json.dump(R.to_dict("records"),open(OUT/"phase4_summary.json","w"),indent=2)

,comparison,sigma_a,sigma_b,sigma_e,rho_family,rho_image,ratio_exact,ratio_2comp,ratio_observed
0,C4 vs C2,0.0441,0.0396,0.1257,0.1007,0.0813,8.3702,10.5756,8.234
1,C4 vs C5,0.2135,0.3042,0.8449,0.0535,0.1086,5.7357,7.6472,5.893



CROSSED DECOMPOSITION

  C4 vs C2
    sigma_a 0.0441   sigma_b 0.0396   sigma_e 0.1257
    rho_family (crossed) 0.1007   rho_image 0.0813
    ratio -- exact 8.37 | two-component 10.58 | observed 8.23

  C4 vs C5
    sigma_a 0.2135   sigma_b 0.3042   sigma_e 0.8449
    rho_family (crossed) 0.0535   rho_image 0.1086
    ratio -- exact 5.74 | two-component 7.65 | observed 5.89

rho_image ranges to 0.1086
  -> material. The reported rho is a LOWER BOUND on the family ICC, because the
     image variance sat in its denominator. Report rho_family as the headline and
     keep the two-component value for comparability, saying which is which.
     This makes the effect LARGER than the manuscript currently claims.


---

## What to write

**If $\rho_{\rm image}$ is negligible.** One sentence in Section 5.3: *"Fitting the crossed model
gives $\sigma_b \approx 0$, so the two-component reduction used throughout is exact here."*
Tables 5--7 stand unchanged.

**If $\rho_{\rm image}$ is material.** The reported $\rho$ values are a **lower bound** on the
family ICC. Report $\rho_{\rm family}$ from the crossed fit as the headline, keep the
two-component value for comparability with the cluster-robust literature, and state which is
which. This strengthens the paper --- the effect is larger than reported, not smaller.

**Either way**, add $\sigma_b$ to Table 5 and relabel its $\sigma_e$ column, which under the
two-component reduction estimates $\sqrt{F\sigma_b^2+\sigma_e^2}$ rather than the residual
standard deviation.

**Check Fig. 3 before reusing it.** If $\rho_{\rm family}$ rises, the number of families needed
for power falls, and the figure's 46 was pessimistic.